# task_03 Solution

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

base = Path("../data")


In [ ]:

admissions = pd.read_csv(base / "admissions.csv")
labs = pd.read_csv(base / "labs.csv")
meds = pd.read_csv(base / "meds.csv")

admissions["admit_date"] = pd.to_datetime(admissions["admit_date"])
admissions["discharge_date"] = pd.to_datetime(admissions["discharge_date"])
labs["lab_date"] = pd.to_datetime(labs["lab_date"])

sorted_adm = admissions.sort_values(["patient_id", "admit_date"])
readmissions = 0
for _, group in sorted_adm.groupby("patient_id"):
    prev_discharge = None
    for row in group.itertuples(index=False):
        if prev_discharge is not None and (row.admit_date - prev_discharge).days <= 14:
            readmissions += 1
        prev_discharge = row.discharge_date
q1 = str(readmissions)

admissions["los_days"] = (admissions["discharge_date"] - admissions["admit_date"]).dt.days
q2 = str(admissions.groupby("diagnosis")["los_days"].mean().sort_values(ascending=False).index[0])

low_sodium = set(labs.loc[labs["sodium"] < 135, "admission_id"])
m03 = set(meds.loc[meds["med_code"] == "M03", "admission_id"])
q3 = f"{(100 * len(low_sodium & m03) / admissions['admission_id'].nunique()):.1f}%"

q4 = str(admissions.groupby("ward")["los_days"].sum().sort_values(ascending=False).index[0])

creat = labs.sort_values(["admission_id", "lab_date"]).groupby("admission_id")["creatinine"].agg(["first", "last"])
creat["increase"] = creat["last"] - creat["first"]
q5 = str(creat.reset_index().sort_values(["increase", "admission_id"], ascending=[False, True]).iloc[0]["admission_id"])


Q1: How many admissions are readmissions occurring within 14 days of the same patient's previous discharge?

In [ ]:
print(q1)


Q2: Which diagnosis has the highest average length of stay?

In [ ]:
print(q2)


Q3: What percentage of admissions had at least one sodium value below 135 and also had medication code M03 during that admission, rounded to 1 decimal place?

In [ ]:
print(q3)


Q4: Which ward has the most total inpatient-days?

In [ ]:
print(q4)


Q5: Which admission_id had the largest increase in creatinine from the first lab to the last lab within that admission? Break ties by the smallest admission_id.

In [ ]:
print(q5)
